In [1]:
from enum import IntEnum
from typing import Optional
import random

In [2]:
class Square(IntEnum):
    a4 = 0
    b4 = 1
    c4 = 2
    d4 = 3
    a3 = 4
    b3 = 5
    c3 = 6
    d3 = 7
    a2 = 8
    b2 = 9
    c2 = 10
    d2 = 11
    a1 = 12
    b1 = 13
    c1 = 14
    d1 = 15

    def __str__(self):
        rank = 4 - (self // 4)
        file = self % 4
        return "abcd"[file] + str(rank)

class Piece(IntEnum):
    K = 0
    W = 1
    M = 2
    F = 3
    P = 4
    k = 5
    w = 6
    m = 7
    f = 8
    p = 9

class Side(IntEnum):
    White = 0
    Black = 1
    Both = 2

ASCII_PIECES = ["K", "W", "M", "F", "P", "k", "w", "m", "f", "p"]


In [3]:
BitBoard = int

def get_bit(bitboard: BitBoard, square: int) -> BitBoard:
    return bitboard & (1 << square)

def set_bit(bitboard: BitBoard, square: int) -> BitBoard:
    return bitboard | (1 << square)

def pop_bit(bitboard: BitBoard, square: int) -> BitBoard:
    return bitboard & ~(1 << square)

def print_bitboard(bitboard: BitBoard) -> None:
    print()
    for rank in range(4):
        for file in range(4):
            square = rank * 4 + file
            if file == 0:
                print(f"  {4-rank} ", end="")
            print(f" {int(bool(get_bit(bitboard, square)))}", end = "")
        print()
    print("\n     a b c d")

def get_ls1b_index(bitboard: BitBoard) -> Optional[int]:
    if bitboard:
        return ((bitboard & -bitboard) - 1).bit_count()
    else:
        return None

In [4]:
# pawn attacks

U16_MAX = 1 << 16
NOT_A_FILE = 0b1110_1110_1110_1110
NOT_D_FILE = 0b0111_0111_0111_0111
NOT_AB_FILE = 0b1100_1100_1100_1100
NOT_CD_FILE = 0b0011_0011_0011_0011

def mask_pawn_attacks(side: int, square: int) -> BitBoard:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if side == Side.White: # White
        if ((bitboard >> 3) & NOT_A_FILE): attacks |= (bitboard >> 3)
        if ((bitboard >> 5) & NOT_D_FILE): attacks |= (bitboard >> 5)
    else:
        if ((bitboard << 3) & NOT_D_FILE): attacks |= (bitboard << 3)
        if ((bitboard << 5) & NOT_A_FILE): attacks |= (bitboard << 5)
    return attacks

def mask_king_attacks(square: int) -> BitBoard:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if (bitboard >> 4): attacks |= (bitboard >> 4)
    if ((bitboard >> 5) & NOT_D_FILE): attacks |= (bitboard >> 5)
    if ((bitboard >> 3) & NOT_A_FILE): attacks |= (bitboard >> 3)
    if ((bitboard >> 1) & NOT_D_FILE): attacks |= (bitboard >> 1)
    if (bitboard << 4): attacks |= (bitboard << 4) % U16_MAX 
    if ((bitboard << 5) & NOT_A_FILE): attacks |= (bitboard << 5) % U16_MAX
    if ((bitboard << 3) & NOT_D_FILE): attacks |= (bitboard << 3) % U16_MAX
    if ((bitboard << 1) & NOT_A_FILE): attacks |= (bitboard << 1) % U16_MAX
    
    return attacks

def mask_wazir_attacks(square: int) -> BitBoard:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if (bitboard >> 4): attacks |= (bitboard >> 4)
    if ((bitboard >> 1) & NOT_D_FILE): attacks |= (bitboard >> 1)
    if (bitboard << 4): attacks |= (bitboard << 4) % U16_MAX 
    if ((bitboard << 1) & NOT_A_FILE): attacks |= (bitboard << 1) % U16_MAX 
    
    return attacks

def mask_ferz_attacks(square: int) -> BitBoard:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if ((bitboard >> 5) & NOT_D_FILE): attacks |= (bitboard >> 5)
    if ((bitboard >> 3) & NOT_A_FILE): attacks |= (bitboard >> 3)
    if ((bitboard << 5) & NOT_A_FILE): attacks |= (bitboard << 5) % U16_MAX 
    if ((bitboard << 3) & NOT_D_FILE): attacks |= (bitboard << 3) % U16_MAX 
    
    return attacks

def generate_ma_attacks_on_the_fly(square: int, block_mask: BitBoard) -> BitBoard:
    attacks = 0
    
    tr = square // 4
    tf = square % 4

    deltas = {
        (0, 1): [(1, 2), (-1, 2)],
        (1, 0): [(2, 1), (2, -1)],
        (0, -1): [(1, -2), (-1, -2)],
        (-1, 0): [(-2, -1), (-2, 1)]
    }
    for (block_dr, block_df), attack_list in deltas.items():
        block_r = tr + block_dr
        block_f = tf + block_df
        block_square = 4 * block_r + block_f 

        if 0 <= block_square and block_square < 16 and (1 << (block_square) & block_mask):
            continue
        for dr, df in attack_list:
            r = tr + dr
            f = tf + df
            if 0 <= r and r < 4 and 0 <= f and f < 4:
                attacks |= 1 << (4 * r + f)
    return attacks


In [ ]:
KING_ATTACKS: list[BitBoard] = [0 for _ in range(16)]
WAZIR_ATTACKS: list[BitBoard] = [0 for _ in range(16)]
FERZ_ATTACKS: list[BitBoard] = [0 for _ in range(16)]
PAWN_ATTACKS: list[list[BitBoard]] = [[0 for _ in range(16)], [0 for _ in range(16)]]

for square in range(16):
    KING_ATTACKS[square] = mask_king_attacks(square)
    WAZIR_ATTACKS[square] = mask_wazir_attacks(square)
    FERZ_ATTACKS[square] = mask_ferz_attacks(square)
    PAWN_ATTACKS[0][square] = mask_pawn_attacks(0, square)
    PAWN_ATTACKS[1][square] = mask_pawn_attacks(1, square)

def print_array(array):
    print("[")
    for x in array:
        print(f"\tBitBoard(0b{x:016b}),")
    print("];")

# print("King Attacks")
# print_array(KING_ATTACKS)
print("Wazir Attacks")
print_array(WAZIR_ATTACKS)
print("Ferz Attacks")
print_array(FERZ_ATTACKS)
print("White Pawn Attacks")
print_array(PAWN_ATTACKS[0])
print("Black Pawn Attacks")
print_array(PAWN_ATTACKS[1])

[
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000010),
	BitBoard(0b0000000000000101),
	BitBoard(0b0000000000001010),
	BitBoard(0b0000000000000100),
	BitBoard(0b0000000000100000),
	BitBoard(0b0000000001010000),
	BitBoard(0b0000000010100000),
	BitBoard(0b0000000001000000),
	BitBoard(0b0000001000000000),
	BitBoard(0b0000010100000000),
	BitBoard(0b0000101000000000),
	BitBoard(0b0000010000000000),
];
[
	BitBoard(0b0000000000100000),
	BitBoard(0b0000000001010000),
	BitBoard(0b0000000010100000),
	BitBoard(0b0000000001000000),
	BitBoard(0b0000001000000000),
	BitBoard(0b0000010100000000),
	BitBoard(0b0000101000000000),
	BitBoard(0b0000010000000000),
	BitBoard(0b0010000000000000),
	BitBoard(0b0101000000000000),
	BitBoard(0b1010000000000000),
	BitBoard(0b0100000000000000),
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000000),
	BitBoard(0b0000000000000000),
]

In [ ]:
def generate_magic():
    return random.randint(0, 2**15-1) & random.randint(0, 2**15-1) & random.randint(0, 2**15-1) & random.randint(0, 2**15-1)

def find_magic_number(square: Square) -> Optional[int]:
    blocking_patterns = []
    subset = 0
    blockers = mask_wazir_attacks(square)
    while True:
        subset = (subset - blockers) & blockers
        blocking_patterns.append(subset)

        if subset == 0:
            break

    attacks = []
    for pattern in blocking_patterns:
        attacks.append(generate_ma_attacks_on_the_fly(square, pattern))

    while True:
        magic = generate_magic()
        used_attacks = [0 for _ in range(4)] # there are at most 4 attack squares for a Ma. Only 2 blockers are relevant. There are hence 2**2 unique attacks patterns per square
        fail = False
        for idx, pattern in enumerate(blocking_patterns):
            magic_index = (((pattern * magic) % U16_MAX) >> (16 - 2))  

            if used_attacks[magic_index] == 0:
                used_attacks[magic_index] = attacks[idx]
            elif used_attacks[magic_index] == attacks[idx]: # constructive collision: blocker configs that map to the same attack pattern (i.e. non-injectively)
                pass
            else:
                fail = True
                break
        if not fail:
            return magic

random.seed(0)
MA_ATTACKS = [[0 for _ in range(4)] for _ in range(16)]
MAGICS = [] 
for rank in range(4):
    for file in range(4):
        square = 4 * rank + file
        magic = find_magic_number(square)
        MAGICS.append(magic)
        blockers = mask_wazir_attacks(square) 

        subset = 0
        while True:
            subset = (subset - blockers) & blockers
            magic_index = (((subset * magic) % U16_MAX) >> (16 - 2))
            MA_ATTACKS[square][magic_index] = generate_ma_attacks_on_the_fly(square, subset)

            if subset == 0:
                break

print("Ma Attacks")
print(MA_ATTACKS)
print("Magics")
print(MAGICS)

[[576, 64, 512, 0], [1408, 128, 1280, 0], [2576, 2560, 16, 0], [1056, 32, 1024, 0], [9220, 8192, 1028, 0], [22536, 2056, 20480, 0], [41217, 257, 40960, 0], [16898, 514, 16384, 0], [16450, 2, 16448, 0], [32901, 5, 32896, 0], [4122, 10, 4112, 0], [8228, 8224, 4, 0], [1056, 32, 1024, 0], [2128, 80, 2048, 0], [416, 256, 160, 0], [576, 512, 64, 0]]
[17424, 8768, 8724, 8320, 4736, 544, 1042, 521, 2080, 1040, 544, 160, 130, 65, 20, 8202]
